# Tier-1 LSTM Variational Autoencoder Agents (Multi-Modality)
This notebook implements the Unsupervised LSTM-VAE strategy.

**Core Architecture Details:**
- **4 Independent LSTM-VAEs** (Logon, Email, File, HTTP).
- **Deterministic Train/Val/Test Split:** 
  - Train: First 800 users (Normal behavior only)
  - Validation: Next 100 users (Used for Hyperparameter & Threshold Tuning)
  - Test: Final 100 users (Used ONLY for final evaluation)
- **MLflow Tracking:** Logs hyperparameters, metrics, and artifacts (ROC, Confusion Matrix).
- **Ablation Study:** Grid search over `latent_dim` (2, 4, 8) and `sequence_length` (7, 14).
- **Regularization:** KL Annealing, Gradient Clipping, Dropout, Weight Decay.
- **Anomaly Scoring:** We export BOTH Reconstruction-only scores and ELBO scores.


In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score, confusion_matrix, roc_curve, ConfusionMatrixDisplay
import mlflow
import itertools
import gc
from pathlib import Path
import os
import matplotlib.pyplot as plt

# Dynamic Path Resolution

# === Determinism: seed all RNGs for reproducible results (as in week3/tier1_score_generation.ipynb) ===
import random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Use a seeded generator for DataLoader shuffle
g = torch.Generator()
g.manual_seed(SEED)

print(f"All RNGs seeded with SEED={SEED}")

ROOT_DIR = Path.cwd()
while not (ROOT_DIR / "pyproject.toml").exists() and ROOT_DIR.parent != ROOT_DIR:
    ROOT_DIR = ROOT_DIR.parent

DATA_FILE = ROOT_DIR / "data" / "dayr4.2.csv"
MLFLOW_DB = f"sqlite:///{ROOT_DIR / 'mlflow.db'}"

mlflow.set_tracking_uri(MLFLOW_DB)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


In [ ]:
print("Establishing Global User Split...")
df_users = pd.read_csv(DATA_FILE, usecols=['user'])
all_unique_users = sorted(df_users['user'].unique())

TRAIN_USERS = set(all_unique_users[:800])
VAL_USERS = set(all_unique_users[800:900])
TEST_USERS = set(all_unique_users[900:1000])

print(f"Train Users: {len(TRAIN_USERS)} | Val Users: {len(VAL_USERS)} | Test Users: {len(TEST_USERS)}")
del df_users
gc.collect()


In [ ]:
class SequenceDataset(Dataset):
    def __init__(self, df, feature_cols, scaler, seq_len=14):
        self.features = scaler.transform(df[feature_cols].fillna(0).astype(np.float32))
        self.labels = (df['insider'].values > 0).astype(int)
        self.users = df['user'].values
        self.weeks = df['week'].values
        self.days = df['day'].values
        self.seq_len = seq_len
        self.indices = []
        
        user_counts = df['user'].value_counts(sort=False)
        current_idx = 0
        for count in user_counts:
            if count >= seq_len:
                for i in range(count - seq_len + 1):
                    self.indices.append(current_idx + i)
            current_idx += count

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        start_idx = self.indices[idx]
        seq = self.features[start_idx : start_idx + self.seq_len]
        label = self.labels[start_idx : start_idx + self.seq_len].max()
        end_idx = start_idx + self.seq_len - 1
        return torch.tensor(seq), torch.tensor(seq), torch.tensor(label, dtype=torch.float32), self.users[end_idx], self.weeks[end_idx], self.days[end_idx]

class LSTM_VAE(nn.Module):
    def __init__(self, input_dim, hidden_dim=16, latent_dim=8, dropout=0.2):
        super(LSTM_VAE, self).__init__()
        self.encoder = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)
        
        self.fc_dec = nn.Linear(latent_dim, hidden_dim)
        self.decoder = nn.LSTM(hidden_dim, input_dim, batch_first=True)
        
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        _, (h, _) = self.encoder(x)
        h = self.dropout(h[-1])
        
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        z = self.reparameterize(mu, logvar)
        
        h_dec = self.fc_dec(z)
        seq_len = x.shape[1]
        h_repeated = h_dec.unsqueeze(1).repeat(1, seq_len, 1)
        
        out, _ = self.decoder(h_repeated)
        return out, mu, logvar


In [ ]:
modalities = ['logon', 'email', 'file_device', 'http']

param_grid = {
    'hidden_dim': [16],
    'latent_dim': [2, 4, 8],
    'sequence_length': [7, 14]
}

fixed_params = {
    'learning_rate': 0.001,
    'dropout': 0.2,
    'weight_decay': 1e-5,
    'epochs': 20, 
    'optimizer': 'Adam'
}

all_test_predictions = []

def get_optimal_threshold(scores, labels):
    best_f1 = 0
    best_thresh = 0
    for p in np.linspace(90, 99.9, 100):
        thresh = np.percentile(scores, p)
        preds = (scores > thresh).astype(int)
        f1 = f1_score(labels, preds, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_thresh = thresh
    if best_thresh == 0:
        best_thresh = np.percentile(scores, 95)
    return best_thresh

for modality in modalities:
    print(f"\n{'='*50}")
    print(f"STARTING VAE ABLATION STUDY FOR {modality.upper()}")
    print(f"{'='*50}")
    
    mlflow.set_experiment(f"VAE_{modality.capitalize()}")
    
    with open(DATA_FILE, 'r') as f:
        all_cols = f.readline().strip().split(',')
        
    feature_cols = [c for c in all_cols if modality in c or c in ['O', 'C', 'E', 'A', 'N', 'role']]
    usecols = ['user', 'day', 'week', 'insider'] + feature_cols
    
    print(f"Loading {len(feature_cols)} features for {modality}...")
    df = pd.read_csv(DATA_FILE, usecols=usecols)
    df = df.sort_values(by=['user', 'day'])
    
    train_df = df[df['user'].isin(TRAIN_USERS)]
    val_df = df[df['user'].isin(VAL_USERS)]
    test_df = df[df['user'].isin(TEST_USERS)]
    
    train_normal = train_df[train_df['insider'] == 0]
    
    scaler = StandardScaler()
    scaler.fit(train_normal[feature_cols].fillna(0).astype(np.float32))
    
    best_val_f1 = -1
    best_val_auc = -1
    best_model_state = None
    best_params = None
    best_threshold_elbo = 0
    best_threshold_recon = 0
    best_seq_len = 14
    
    keys, values = zip(*param_grid.items())
    combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]
    
    for run_idx, params in enumerate(combinations):
        seq_len = params['sequence_length']
        hidden_dim = params['hidden_dim']
        latent_dim = params['latent_dim']
        
        print(f"\n--- Run {run_idx+1}/{len(combinations)} | Seq: {seq_len}, Latent: {latent_dim} ---")
        
        with mlflow.start_run():
            mlflow.log_params(params)
            mlflow.log_params(fixed_params)
            
            train_dataset = SequenceDataset(train_normal, feature_cols, scaler, seq_len=seq_len)
            train_loader = DataLoader(train_dataset, batch_size=1024, shuffle=True, generator=g, num_workers=0, pin_memory=True)
            
            val_dataset = SequenceDataset(val_df, feature_cols, scaler, seq_len=seq_len)
            val_loader = DataLoader(val_dataset, batch_size=1024, shuffle=False, num_workers=0, pin_memory=True)
            
            model = LSTM_VAE(input_dim=len(feature_cols), hidden_dim=hidden_dim, latent_dim=latent_dim, dropout=fixed_params['dropout']).to(device)
            optimizer = optim.Adam(model.parameters(), lr=fixed_params['learning_rate'], weight_decay=fixed_params['weight_decay'])
            
            best_val_elbo = float('inf')
            patience = 3
            patience_counter = 0
            
            for epoch in range(fixed_params['epochs']):
                model.train()
                # KL Annealing (smoother schedule)
                kl_weight = min(1.0, (epoch + 1) / 20.0)
                
                total_loss, total_recon, total_kld = 0, 0, 0
                mu_means, mu_stds, logvar_means = [], [], []
                
                for batch_x, batch_y, _, _, _, _ in train_loader:
                    batch_x, batch_y = batch_x.to(device), batch_y.to(device)
                    optimizer.zero_grad()
                    out, mu, logvar = model(batch_x)
                    
                    recon_loss = F.mse_loss(out, batch_y, reduction='mean')
                    kld_loss = -0.5 * torch.mean(torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1))
                    
                    loss = recon_loss + (kl_weight * kld_loss)
                    loss.backward()
                    
                    # Gradient clipping
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    optimizer.step()
                    
                    total_loss += loss.item()
                    total_recon += recon_loss.item()
                    total_kld += kld_loss.item()
                    
                    mu_means.append(mu.mean().item())
                    mu_stds.append(mu.std().item())
                    logvar_means.append(logvar.mean().item())
                
                # Validation Early Stopping check
                model.eval()
                val_total_elbo = 0
                with torch.no_grad():
                    for batch_x, _, _, _, _, _ in val_loader:
                        batch_x = batch_x.to(device)
                        out, mu, logvar = model(batch_x)
                        recon_loss = F.mse_loss(out, batch_x, reduction='mean')
                        kld_loss = -0.5 * torch.mean(torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1))
                        val_total_elbo += (recon_loss + kld_loss).item()
                
                avg_val_elbo = val_total_elbo / len(val_loader)
                
                if avg_val_elbo < best_val_elbo:
                    best_val_elbo = avg_val_elbo
                    patience_counter = 0
                else:
                    patience_counter += 1
                
                if patience_counter >= patience:
                    print(f"Early stopping at epoch {epoch+1}")
                    break
                    
            mlflow.log_metric("train_total_loss", total_loss / len(train_loader))
            mlflow.log_metric("train_recon_loss", total_recon / len(train_loader))
            mlflow.log_metric("train_kld_loss", total_kld / len(train_loader))
            mlflow.log_metric("kl_weight", kl_weight)
            mlflow.log_metric("mu_mean", np.mean(mu_means))
            mlflow.log_metric("mu_std", np.mean(mu_stds))
            mlflow.log_metric("logvar_mean", np.mean(logvar_means))
            
            # Validation Metric Evaluation
            model.eval()
            val_recon_scores, val_elbo_scores, val_labels = [], [], []
            with torch.no_grad():
                for batch_x, _, labels, _, _, _ in val_loader:
                    batch_x = batch_x.to(device)
                    out, mu, logvar = model(batch_x)
                    
                    # Score 1: Recon MSE per sample
                    mse = torch.mean((out - batch_x)**2, dim=[1, 2])
                    val_recon_scores.extend(mse.cpu().numpy())
                    
                    # Score 2: ELBO per sample (Recon + KL)
                    kld = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1)
                    elbo = mse + kld # Beta=1 for now
                    val_elbo_scores.extend(elbo.cpu().numpy())
                    
                    val_labels.extend(labels.numpy())
            
            # We tune threshold and monitor F1 based on ELBO as it's the more robust metric
            thresh_elbo = get_optimal_threshold(np.array(val_elbo_scores), val_labels)
            preds_elbo = (np.array(val_elbo_scores) > thresh_elbo).astype(int)
            val_f1 = f1_score(val_labels, preds_elbo, zero_division=0)
            val_auc = roc_auc_score(val_labels, val_elbo_scores)
            
            mlflow.log_metric("val_auc", val_auc)
            mlflow.log_metric("val_f1", val_f1)
            
            print(f"Val AUC: {val_auc:.4f} | Val F1: {val_f1:.4f} | Thresh: {thresh_elbo:.4f}")
            
            if val_f1 > best_val_f1 or (val_f1 == best_val_f1 and val_auc > best_val_auc):
                best_val_f1 = val_f1
                best_val_auc = val_auc
                best_model_state = {k: v.cpu() for k, v in model.state_dict().items()}
                best_params = params
                best_threshold_elbo = thresh_elbo
                best_threshold_recon = get_optimal_threshold(np.array(val_recon_scores), val_labels)
                best_seq_len = seq_len
                
    # --- EVALUATE BEST MODEL ON TEST SET ---
    print(f"\n>> Best Params for {modality.upper()}: {best_params} | Val F1: {best_val_f1:.4f}")
    
    best_model = LSTM_VAE(input_dim=len(feature_cols), hidden_dim=best_params['hidden_dim'], latent_dim=best_params['latent_dim'], dropout=fixed_params['dropout']).to(device)
    best_model.load_state_dict(best_model_state)
    best_model.eval()
    
    test_dataset = SequenceDataset(test_df, feature_cols, scaler, seq_len=best_seq_len)
    test_loader = DataLoader(test_dataset, batch_size=1024, shuffle=False, num_workers=0, pin_memory=True)
    
    test_recon, test_elbo, test_labels = [], [], []
    test_users, test_weeks, test_days = [], [], []
    
    with torch.no_grad():
        for batch_x, _, labels, u, w, d in test_loader:
            batch_x = batch_x.to(device)
            out, mu, logvar = best_model(batch_x)
            
            mse = torch.mean((out - batch_x)**2, dim=[1, 2])
            kld = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1)
            elbo = mse + kld
            
            test_recon.extend(mse.cpu().numpy())
            test_elbo.extend(elbo.cpu().numpy())
            test_labels.extend(labels.numpy())
            test_users.extend(u)
            test_weeks.extend(w.numpy())
            test_days.extend(d.numpy())
            
    test_preds_elbo = (np.array(test_elbo) > best_threshold_elbo).astype(int)
    test_auc_elbo = roc_auc_score(test_labels, test_elbo)
    test_f1_elbo = f1_score(test_labels, test_preds_elbo, zero_division=0)
    
    test_preds_recon = (np.array(test_recon) > best_threshold_recon).astype(int)
    test_auc_recon = roc_auc_score(test_labels, test_recon)
    test_f1_recon = f1_score(test_labels, test_preds_recon, zero_division=0)
    
    print(f"TEST ELBO SCORES  -> AUC: {test_auc_elbo:.4f} | F1: {test_f1_elbo:.4f}")
    print(f"TEST RECON SCORES -> AUC: {test_auc_recon:.4f} | F1: {test_f1_recon:.4f}")
    
    with mlflow.start_run(run_name=f"{modality}_best_test"):
        mlflow.log_params(best_params)
        mlflow.log_params(fixed_params)
        mlflow.log_metric("test_auc_elbo", test_auc_elbo)
        mlflow.log_metric("test_f1_elbo", test_f1_elbo)
        mlflow.log_metric("test_auc_recon", test_auc_recon)
        mlflow.log_metric("test_f1_recon", test_f1_recon)
        mlflow.set_tag("is_best_model", "true")
        
    def normalize_to_prob(scores, threshold, steepness=10):
        return 1 / (1 + np.exp(-(scores - threshold) * steepness))
        
    df_preds = pd.DataFrame({
        'user': test_users,
        'week': test_weeks,
        'day': test_days,
        'label': test_labels,
        f'{modality}_recon_score': normalize_to_prob(np.array(test_recon), best_threshold_recon),
        f'{modality}_elbo_score': normalize_to_prob(np.array(test_elbo), best_threshold_elbo)
    })
    
    df_preds = df_preds.groupby(['user', 'week', 'day', 'label'], as_index=False).max()
    all_test_predictions.append(df_preds)
    
    del df, train_df, val_df, test_df, train_normal
    gc.collect()

print("\nMerging final scores...")
dfs_indexed = [df.set_index(['user', 'week', 'day', 'label']) for df in all_test_predictions]
tier1_scores = pd.concat(dfs_indexed, axis=1).reset_index()
tier1_scores.fillna(0, inplace=True)
out_path = ROOT_DIR / "week3" / "tier1_vae_scores.csv"
tier1_scores.to_csv(out_path, index=False)
print(f"Saved tier1_vae_scores.csv to {out_path}")
